
# Guided Project 3: AI Financial Report Analyst

## Business Scenario

A financial advisory company regularly receives quarterly and annual reports (PDFs) from multiple firms. Analysts spend significant time reviewing these documents to extract key insights such as revenue trends, expenses, risks, and future outlooks.

Clients often ask multi-turn, contextual questions, for example:

* *“What was the revenue growth this quarter compared to last?”*
* *“What risks did the company highlight?”*
* *“Can you summarize all risks and growth strategies we discussed earlier?”*

Currently, this process is time-consuming and inconsistent across analysts.

The company wants an **AI-powered Financial Report Analyst** that can automatically process financial documents, summarize insights, and remember prior client questions for more natural and effective interactions.



## Problem Statement

Develop an **AI Financial Report Analyst using LangChain** that can:

1. Load and process financial report PDFs.
2. Use **Prompt Templates** to extract structured insights on Revenue, Expenses, Risks, and Outlook.
3. Enable **multi-turn conversations with memory**, so the system recalls prior questions and answers.
4. Provide **consistent, finance-domain responses** tailored to client queries.



## Step-by-Step Approach

### Step 1: Setup

* Install required libraries: `langchain`, `pypdf`.
* Import key modules: `PyPDFLoader`, `PromptTemplate`, `InMemoryChatMessageHistory`, `LangChain's Chain component`.

### Step 2: Load and Ingest Financial Reports

* Load the PDF report using **PyPDFLoader**.
* Extract text from the document.
* Optionally split text into chunks using **RecursiveCharacterTextSplitter**.
* For demonstration, use a limited set of pages.

### Step 3: Define Prompt Template

* Create a structured prompt that instructs the AI to:

  * Identify **Revenue Trends**.
  * Summarize **Expenses**.
  * List **Risks**.
  * Highlight **Future Outlook/Strategies**.
* Ensure outputs are concise and presented in **bullet-style format**.

### Step 4: Add Conversation Memory

* Use **InMemoryChatMessageHistory** to store prior user questions and responses.
* Example flow:

  * User: *“Summarize the revenue growth trend.”*
  * User: *“Now tell me about the risks mentioned.”*
  * User: *“Combine both insights into a single summary.”*
* The AI should recall earlier responses and merge them into a consolidated answer.

### Step 5: Build the LLM Chain

* Connect:

  * Bedrock LLM (for text generation).
  * The Prompt Template (for structured instructions).
  * Conversation Memory (for context persistence).
* Use **LangChain's Chain component** to link all components.

### Step 6: Simulate Client Queries

* Demonstrate the system with sample queries:

  1. *“What is the company’s revenue growth this year?”* → Extracted from the financial report.
  2. *“What risks were mentioned?”* → AI summarizes reported risks.
  3. *“Give me a combined summary of revenue and risks so far.”* → AI recalls earlier answers and provides a consolidated summary.


### Load all the necessary libraries / packages

In [1]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory

In [2]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content

'Hello! How can I help you today?'

In [4]:
# --------------------------
# Document ingestion
# --------------------------
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(r"financial_report.pdf")   # Change to your PDF path
documents = loader.load()

# Split into chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)
texts = text_splitter.split_documents(documents)

texts

2026-03-18 04:31:46.288638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/labuser/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-09T04:12:40+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-09-09T04:12:40+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'financial_report.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Saudi Aramco Financial Report\nCompany Snapshot\nSaudi Aramco (Saudi Arabian Oil Company) is the world’s largest integrated oil and gas company,\nheadquartered in Dhahran, Saudi Arabia. The company is majority-owned by the Government of\nSaudi Arabia. CEO: Amin H. Nasser. Operations include exploration, production, refining,\ndistribution, and petrochemicals.\nFY 2024 Financial Highlights\nMetric\nValue (USD Billion)\nRevenue\n480.45\nOperating Income\n206.57\nNet Income\n106.25\nTotal Assets\n646.30\nShareholder Equity\n440.36\nDividend Performance\nIn 2024, Aramco declared total dividend pa

In [5]:
# --------------------------
# Embeddings & Vector Store
# --------------------------

from langchain_community.vectorstores import Chroma
from langchain_aws import BedrockEmbeddings
import chromadb.config

embeddings=BedrockEmbeddings(model_id='cohere.embed-english-v3', #amazon.titan-embed-text-v1
                        aws_access_key_id='',
                        aws_secret_access_key='',
                       region_name='us-east-1')


# Create Chroma index
db = Chroma.from_documents(texts, embeddings)

In [6]:
db

In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    # --- System Message ---
    ("system", """
You are a financial report analyst.
Use the retrieved context to answer queries.
If answer for the query is not rooted in the context provided,
answer based on the query appropriately.

Answer any user questions based solely on the context below:

<context>
{context}
</context>
"""),

    # --- Chat History Placeholder ---
    ("placeholder", "{chat_history}"),

    # --- Human Message ---
    ("human", "{input}")
])
#print(chat_prompt)

In [8]:
db.similarity_search("what is Net profit in FY 2024?")

[Document(metadata={'page_label': '1', 'author': '(anonymous)', 'producer': 'ReportLab PDF Library - www.reportlab.com', 'title': '(anonymous)', 'source': 'financial_report.pdf', 'moddate': '2025-09-09T04:12:40+00:00', 'page': 0, 'creationdate': '2025-09-09T04:12:40+00:00', 'trapped': '/False', 'total_pages': 2, 'keywords': '', 'subject': '(unspecified)', 'creator': '(unspecified)'}, page_content='of 2025, base dividends reached USD 42.3 billion while performance-linked dividends dropped\nsignificantly by 98% year-on-year, reflecting lower free cash flow due to weaker oil prices.\nMarket & Industry Context\nNet profit in FY 2024 declined by 12% to USD 106.25 billion due to lower energy prices. In Q2\n2025, net profit fell further by 22% to USD 22.7 billion. Average realized oil prices dropped from\nabout USD 85.7/barrel in Q2 2024 to USD 66.7/barrel in Q2 2025.\nDetailed Financial Analysis\nPeriod\nAdjusted Net Income\nOperating Cash Flow\nFree Cash Flow\nQ2 2025\n24.5 B\n27.5 B\n15.2 

### Augmented Generation

In [9]:
retriever=db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

### Create_stuff_documents_chain
create_stuff_documents_chain is a helper function in LangChain that:

    Takes multiple retrieved documents (from a retriever) as input.
    
    “Stuffs” all the documents together into a single prompt.
    
    Passes that combined text to an LLM with the given prompt template.
    
    Returns a chain that produces an output (answer) from those documents

In [10]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

docs_chain = create_stuff_documents_chain(llm, chat_prompt)

retriever_chain = create_retrieval_chain(
    retriever,
    docs_chain
)

In [11]:
from rich import print
# print(docs_chain)

### Memory

In [12]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [13]:
chain_with_memory = RunnableWithMessageHistory(
    retriever_chain,
    get_session_history,
    input_messages_key="input", #input_messages_key → Matches {input} in the ChatPromptTemplate.
    history_messages_key="chat_history",
    output_messages_key='answer' #history_messages_key → Matches {chat_history} in the ChatPromptTemplate.
)


In [14]:
query="What risks were mentioned?"

In [15]:
session_id="March17"

In [16]:
response = chain_with_memory.invoke(
        {"input": query},
        config={"configurable": {"session_id": session_id}}
    )

In [17]:
print(response)

{
    'input': 'What risks were mentioned?',
    'chat_history': [],
    'context': [
        Document(
            metadata={
                'keywords': '',
                'creator': '(unspecified)',
                'page_label': '2',
                'subject': '(unspecified)',
                'page': 1,
                'title': '(anonymous)',
                'creationdate': '2025-09-09T04:12:40+00:00',
                'total_pages': 2,
                'author': '(anonymous)',
                'producer': 'ReportLab PDF Library - www.reportlab.com',
                'moddate': '2025-09-09T04:12:40+00:00',
                'source': 'financial_report.pdf',
                'trapped': '/False'
            },
            page_content='Financial Strength & Liquidity\nAramco maintains robust cash generation capacity despite 
market volatility. As of H1 2025, gearing\nratio stood at 6.5% with borrowings at USD 92.9 billion. The company 
engaged in asset\nmonetization and plans to issue Islamic bonds to strengthen liquidity.\nChallenges & Outlook\nKey
challenges include sustained oil price volatility and dividend sustainability pressures.\nPerformance-linked 
dividends dropped by 98% in H1 2025. Capital expenditure guidance for 2025\nremains at USD 52–58 billion, with 
expectations of demand growth in the second half of the year.'
        ),
        Document(
            metadata={
                'trapped': '/False',
                'page': 0,
                'page_label': '1',
                'creator': '(unspecified)',
                'source': 'financial_report.pdf',
                'keywords': '',
                'title': '(anonymous)',
                'moddate': '2025-09-09T04:12:40+00:00',
                'author': '(anonymous)',
                'creationdate': '2025-09-09T04:12:40+00:00',
                'total_pages': 2,
                'producer': 'ReportLab PDF Library - www.reportlab.com',
                'subject': '(unspecified)'
            },
            page_content='of 2025, base dividends reached USD 42.3 billion while performance-linked dividends 
dropped\nsignificantly by 98% year-on-year, reflecting lower free cash flow due to weaker oil prices.\nMarket & 
Industry Context\nNet profit in FY 2024 declined by 12% to USD 106.25 billion due to lower energy prices. In 
Q2\n2025, net profit fell further by 22% to USD 22.7 billion. Average realized oil prices dropped from\nabout USD 
85.7/barrel in Q2 2024 to USD 66.7/barrel in Q2 2025.\nDetailed Financial Analysis\nPeriod\nAdjusted Net 
Income\nOperating Cash Flow\nFree Cash Flow\nQ2 2025\n24.5 B\n27.5 B\n15.2 B\nH1 2025\n50.9 B\n59.3 B\n34.4 B\nFY 
2024\n106.25 B\n-\n-'
        ),
        Document(
            metadata={
                'creator': '(unspecified)',
                'creationdate': '2025-09-09T04:12:40+00:00',
                'page': 0,
                'trapped': '/False',
                'page_label': '1',
                'keywords': '',
                'source': 'financial_report.pdf',
                'producer': 'ReportLab PDF Library - www.reportlab.com',
                'total_pages': 2,
                'author': '(anonymous)',
                'moddate': '2025-09-09T04:12:40+00:00',
                'title': '(anonymous)',
                'subject': '(unspecified)'
            },
            page_content='Saudi Aramco Financial Report\nCompany Snapshot\nSaudi Aramco (Saudi Arabian Oil Company)
is the world’s largest integrated oil and gas company,\nheadquartered in Dhahran, Saudi Arabia. The company is 
majority-owned by the Government of\nSaudi Arabia. CEO: Amin H. Nasser. Operations include exploration, production,
refining,\ndistribution, and petrochemicals.\nFY 2024 Financial Highlights\nMetric\nValue (USD 
Billion)\nRevenue\n480.45\nOperating Income\n206.57\nNet Income\n106.25\nTotal Assets\n646.30\nShareholder 
Equity\n440.36\nDividend Performance\nIn 2024, Aramco declared total dividend payouts of approximately USD 85.4 
billion. In the first 

In [18]:
print(response['answer'])

The report mentions two key challenges and risks for Saudi Aramco: 

1. Sustained Oil Price Volatility: The volatility in oil prices is a significant risk factor, as it directly 
impacts Aramco's revenue and profit. In Q2 2025, the average realized oil price dropped by nearly $20/barrel 
compared to Q2 2024, leading to a decline in net profit. 

2. Dividend Sustainability Pressures: There is pressure on dividend sustainability due to the drop in 
performance-linked dividends. In H1 2025, these dividends decreased by 98% year-on-year because of lower free cash 
flow resulting from weaker oil prices. This could impact investors' expectations and the company's ability to 
attract investment. 

These risks are inherent in the oil and gas industry and are beyond the company's full control. However, Aramco is 
taking steps to mitigate these challenges through asset monetization and plans to issue Islamic bonds

## For Multiple Queries

In [19]:
session_id = "March17"
queries = [
    "What is the company's revenue growth this year 2025?",
    "What risks were mentioned?",
    "Give me a combined summary of revenue and risks so far.",
    "What are my previous queries?"
]

In [20]:
for i, q in enumerate(queries, 1):
    response = chain_with_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"\nQuery {i}: {q}")
    print("Answer:", response["answer"])



Query 1: What is the company's revenue growth this year 2025?

Answer: The context provided does not contain sufficient information to calculate the revenue growth for the full 
year 2025, as it only covers financial data up to the first half of that year (H1 2025). 

However, based on the information available, we can calculate the revenue growth from the first half of 2024 (H1 
2024, if available) to the first half of 2025 (H1 2025). 

Let's calculate the revenue growth for H1 2025: 

> Assuming Revenue for H1 2024 = $240.225 billion (half of the full-year revenue for 2024)
> Revenue for H1 2025 = $291.65 billion (calculated as: Revenue for FY 2024 / 2)

Now, we can

Query 2: What risks were mentioned?

Answer: The report mentions two key challenges and risks for Saudi Aramco: 

1. Sustained Oil Price Volatility: Oil price volatility is a significant challenge for the company as it directly 
impacts Aramco's revenue and profit. In Q2 2025, the average realized oil price dropped to $66.7/barrel, a decline 
of nearly $20/barrel from Q2 2024, which led to a decrease in net profit. Volatile oil prices can make it difficult
for the company to predict and maintain stable financial performance. 

2. Dividend Sustainability Pressures: There are concerns about the sustainability of dividends. In the first half 
of 2025, performance-linked dividends dropped significantly (by 98%) year-on-year due to lower free cash flow 
resulting from weaker oil prices. This could impact investors' expectations and dividend payouts in the future, 
potentially affecting the company's ability to attract and retain investors. 

These

Query 3: Give me a combined summary of revenue and risks so far.

Answer: Here is a combined summary of Saudi Aramco's revenue performance and the risks mentioned in the context 
provided: 

**Revenue Performance:**
- Saudi Aramco, the world's largest integrated oil and gas company, generated strong revenue in FY 2024, totaling 
USD 480.45 billion. 
- However, the company faces challenges in maintaining this performance in 2025 due to market volatility. 
- In the first half of 2025 (H1 2025), the company's revenue is estimated to be USD 291.65 billion (extrapolated 
from FY 2024 figures), representing potential growth from the first half of 2024. 

**Risks Mentioned:**
1. Sustained Oil Price Volatility: Volatile oil prices present a significant risk to Aramco's financial 
performance. In Q2 2025, the average realized

Query 4: What are my previous queries?

Answer: Your previous queries were as follows: 
1. "What risks were mentioned?" 
2. "What is the company's revenue growth this year 2025?" 
3. "What risks were mentioned in the report?" 
4. "Can you calculate the revenue growth for me based on this text?" 
5. "Provide a combined summary of revenue and risks." 

Would you like me to answer any new queries or provide more insights on the topics discussed so far?